# **Loggin, Creating Dataframs in Pandas, Creating Database and inserting tables in SQL**

In [1]:
import pandas as pd
import os
from sqlalchemy import create_engine
import logging
import time

logging.basicConfig(
    filename = "Logs/ingestion_db.log",
    level=logging.DEBUG,
    format="%(asctime)s - %(levelname)s - %(message)s",
    filemode="a"
)

engine = create_engine('sqlite:///vendor_trade.db')

'''INSERT CONTINUOUS DATA FRAME INTO DATABASE TABLE'''
def ingest_db(df, table_name, engine):
    df.to_sql(table_name, con = engine, if_exists = 'replace', index=False,chunksize=100000)

def load_raw_data():
    '''This function will load the CSVs as dataframe and ingest into db'''
    start = time.time()
    for file in os.listdir(r'/Data'):
                           if '.csv' in file:
                               df = pd.read_csv('/Data/' + file)
                               logging.info(f'Ingesting {file} in db')
                               ingest_db(df, file[:-4], engine)

    end = time.time()
    total_time = (end - start)/60

    logging.info('------------Ingestion Complete------------')
    logging.info(f'\nTotal TIme Taken: {total_time} minutes')

if __name__=='__main__':
    load_raw_data()

# ***Exploratary Data Analysis***

This phase focuses on understanding the tables stored in the database to uncover patterns, validate data quality, and identify opportunities for optimization.

Objectives




*   Analyze how data is structured across tables.
*   Determine if aggregated tables are needed for faster and more insightful analysis.



---



---






In [2]:
import sqlite3
import pandas as pd

# creating database connection
conn = sqlite3.connect('vendor_trade.db')

In [3]:
#Let's understand which are the tables in Database.
tables = pd.read_sql("select name from sqlite_master where type = 'table' ",conn)
tables

,name
0,end_inventory
1,sales
2,vendor_invoice
3,begin_inventory
4,purchase_prices
5,purchases


In [4]:
# Let's explore the structure and data types of all tables in the database.
# Objective: Understand the schema and the nature of data stored in each table.

for table in tables['name']:
  print('-'*50, f"{table}",'-'*50)
  print("Count  of records:",pd.read_sql(f"select count(*) from {table}",conn).values[0])
  display(pd.read_sql(f"select * from {table} limit 5", conn))



-------------------------------------------------- end_inventory --------------------------------------------------
Count  of records: [224489]


,InventoryId,Store,City,Brand,Description,Size,onHand,Price,endDate
0,1_HARDERSFIELD_58,1,HARDERSFIELD,58,Gekkeikan Black & Gold Sake,750mL,11,12.99,2024-12-31
1,1_HARDERSFIELD_62,1,HARDERSFIELD,62,Herradura Silver Tequila,750mL,7,36.99,2024-12-31
2,1_HARDERSFIELD_63,1,HARDERSFIELD,63,Herradura Reposado Tequila,750mL,7,38.99,2024-12-31
3,1_HARDERSFIELD_72,1,HARDERSFIELD,72,No. 3 London Dry Gin,750mL,4,34.99,2024-12-31
4,1_HARDERSFIELD_75,1,HARDERSFIELD,75,Three Olives Tomato Vodka,750mL,7,14.99,2024-12-31


-------------------------------------------------- sales --------------------------------------------------
Count  of records: [12825363]


,InventoryId,Store,Brand,Description,Size,SalesQuantity,SalesDollars,SalesPrice,SalesDate,Volume,Classification,ExciseTax,VendorNo,VendorName
0,1_HARDERSFIELD_1004,1,1004,Jim Beam w/2 Rocks Glasses,750mL,1,16.49,16.49,2024-01-01,750.0,1,0.79,12546,JIM BEAM BRANDS COMPANY
1,1_HARDERSFIELD_1004,1,1004,Jim Beam w/2 Rocks Glasses,750mL,2,32.98,16.49,2024-01-02,750.0,1,1.57,12546,JIM BEAM BRANDS COMPANY
2,1_HARDERSFIELD_1004,1,1004,Jim Beam w/2 Rocks Glasses,750mL,1,16.49,16.49,2024-01-03,750.0,1,0.79,12546,JIM BEAM BRANDS COMPANY
3,1_HARDERSFIELD_1004,1,1004,Jim Beam w/2 Rocks Glasses,750mL,1,14.49,14.49,2024-01-08,750.0,1,0.79,12546,JIM BEAM BRANDS COMPANY
4,1_HARDERSFIELD_1005,1,1005,Maker's Mark Combo Pack,375mL 2 Pk,2,69.98,34.99,2024-01-09,375.0,1,0.79,12546,JIM BEAM BRANDS COMPANY


-------------------------------------------------- vendor_invoice --------------------------------------------------
Count  of records: [5543]


...

,InventoryId,Store,City,Brand,Description,Size,onHand,Price,startDate
0,1_HARDERSFIELD_58,1,HARDERSFIELD,58,Gekkeikan Black & Gold Sake,750mL,8,12.99,2024-01-01
1,1_HARDERSFIELD_60,1,HARDERSFIELD,60,Canadian Club 1858 VAP,750mL,7,10.99,2024-01-01
2,1_HARDERSFIELD_62,1,HARDERSFIELD,62,Herradura Silver Tequila,750mL,6,36.99,2024-01-01
3,1_HARDERSFIELD_63,1,HARDERSFIELD,63,Herradura Reposado Tequila,750mL,3,38.99,2024-01-01
4,1_HARDERSFIELD_72,1,HARDERSFIELD,72,No. 3 London Dry Gin,750mL,6,34.99,2024-01-01


-------------------------------------------------- purchase_prices --------------------------------------------------
Count  of records: [12261]


,Brand,Description,Price,Size,Volume,Classification,PurchasePrice,VendorNumber,VendorName
0,58,Gekkeikan Black & Gold Sake,12.99,750mL,750,1,9.28,8320,SHAW ROSS INT L IMP LTD
1,62,Herradura Silver Tequila,36.99,750mL,750,1,28.67,1128,BROWN-FORMAN CORP
2,63,Herradura Reposado Tequila,38.99,750mL,750,1,30.46,1128,BROWN-FORMAN CORP
3,72,No. 3 London Dry Gin,34.99,750mL,750,1,26.11,9165,ULTRA BEVERAGE COMPANY LLP
4,75,Three Olives Tomato Vodka,14.99,750mL,750,1,10.94,7245,PROXIMO SPIRITS INC.


-------------------------------------------------- purchases --------------------------------------------------
Count  of records: [2372474]


,InventoryId,Store,Brand,Description,Size,VendorNumber,VendorName,PONumber,PODate,ReceivingDate,InvoiceDate,PayDate,PurchasePrice,Quantity,Dollars,Classification
0,69_MOUNTMEND_8412,69,8412,Tequila Ocho Plata Fresno,750mL,105,ALTAMAR BRANDS LLC,8124,2023-12-21,2024-01-02,2024-01-04,2024-02-16,35.71,6,214.26,1
1,30_CULCHETH_5255,30,5255,TGI Fridays Ultimte Mudslide,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-01,2024-01-07,2024-02-21,9.35,4,37.40,1
2,34_PITMERDEN_5215,34,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-02,2024-01-07,2024-02-21,9.41,5,47.05,1
3,1_HARDERSFIELD_5255,1,5255,TGI Fridays Ultimte Mudslide,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-01,2024-01-07,2024-02-21,9.35,6,56.10,1
4,76_DONCASTER_2034,76,2034,Glendalough Double Barrel,750mL,388,ATLANTIC IMPORTING COMPANY,8169,2023-12-24,2024-01-02,2024-01-09,2024-02-16,21.32,5,106.60,1




---


Let's also focus on a single vendor to analyze data distribution. (Example VendorNo = 4466)

Objective: This helps us understand existing patterns and identify gaps, guiding which additional derived columns (e.g., margins, ratios) are needed for deeper analysis.

---




In [5]:
purchases_4466 = pd.read_sql("select * from purchases where VendorNumber = 4466", conn)
purchases_4466

,InventoryId,Store,Brand,Description,Size,VendorNumber,VendorName,PONumber,PODate,ReceivingDate,InvoiceDate,PayDate,PurchasePrice,Quantity,Dollars,Classification
0,30_CULCHETH_5255,30,5255,TGI Fridays Ultimte Mudslide,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-01,2024-01-07,2024-02-21,9.35,4,37.40,1
1,34_PITMERDEN_5215,34,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-02,2024-01-07,2024-02-21,9.41,5,47.05,1
2,1_HARDERSFIELD_5255,1,5255,TGI Fridays Ultimte Mudslide,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-01,2024-01-07,2024-02-21,9.35,6,56.10,1
3,38_GOULCREST_5215,38,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8207,2023-12-27,2024-01-07,2024-01-19,2024-02-26,9.41,6,56.46,1
4,59_CLAETHORPES_5215,59,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8207,2023-12-27,2024-01-05,2024-01-19,2024-02-26,9.41,6,56.46,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2187,81_PEMBROKE_5215,81,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,13595,2024-12-20,2024-12-29,2025-01-04,2025-02-10,9.41,6,56.46,1
2188,62_KILMARNOCK_5255,62,5255,TGI Fridays Ultimte Mudslide,1.75L,4466,AMERICAN VINTAGE BEVERAGE,13595,2024-12-20,2024-12-28,2025-01-04,2025-02-10,9.35,5,46.75,1
2189,34_PITMERDEN_5215,34,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,13595,2024-12-20,2024-12-28,2025-01-04,2025-02-10,9.41,5,47.05,1
2190,6_GOULCREST_5215,6,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,13595,2024-12-20,2024-12-31,2025-01-04,2025-02-10,9.41,6,56.46,1


In [6]:
#Lets analyse the total purchases by VendorNo = 4466 for each brand
purchases_4466.groupby('Brand')[["Dollars","Quantity"]].sum()

,Dollars,Quantity
Brand,,
3140,51921.60,4640
5215,46325.43,4923
5255,58110.25,6215


In [17]:
purchase_prices_4466 = pd.read_sql_query("""select * from purchase_prices where VendorNumber = 4466""",conn)
purchase_prices_4466

,Brand,Description,Price,Size,Volume,Classification,PurchasePrice,VendorNumber,VendorName
0,5215,TGI Fridays Long Island Iced,12.99,1750mL,1750,1,9.41,4466,AMERICAN VINTAGE BEVERAGE
1,5255,TGI Fridays Ultimte Mudslide,12.99,1750mL,1750,1,9.35,4466,AMERICAN VINTAGE BEVERAGE
2,3140,TGI Fridays Orange Dream,14.99,1750mL,1750,1,11.19,4466,AMERICAN VINTAGE BEVERAGE


In [18]:
vendor_invoice_4466 = pd.read_sql_query("""select * from vendor_invoice where VendorNumber = 4466""",conn)
vendor_invoice_4466.head(10)

,VendorNumber,VendorName,InvoiceDate,PONumber,PODate,PayDate,Quantity,Dollars,Freight,Approval
0,4466,AMERICAN VINTAGE BEVERAGE,2024-01-07,8137,2023-12-22,2024-02-21,15,140.55,8.57,None
1,4466,AMERICAN VINTAGE BEVERAGE,2024-01-19,8207,2023-12-27,2024-02-26,335,3142.33,16.97,None
2,4466,AMERICAN VINTAGE BEVERAGE,2024-01-18,8307,2024-01-03,2024-02-18,41,383.35,1.99,None
3,4466,AMERICAN VINTAGE BEVERAGE,2024-01-27,8469,2024-01-14,2024-03-11,72,673.20,3.30,None
4,4466,AMERICAN VINTAGE BEVERAGE,2024-02-04,8532,2024-01-19,2024-03-15,79,740.21,3.48,None
5,4466,AMERICAN VINTAGE BEVERAGE,2024-02-09,8604,2024-01-24,2024-03-15,347,3261.37,17.61,None
6,4466,AMERICAN VINTAGE BEVERAGE,2024-02-17,8793,2024-02-05,2024-04-02,72,675.36,3.17,None
7,4466,AMERICAN VINTAGE BEVERAGE,2024-03-01,8892,2024-02-12,2024-03-28,117,1096.05,5.15,None
8,4466,AMERICAN VINTAGE BEVERAGE,2024-03-07,8995,2024-02-19,2024-04-02,129,1209.27,5.44,None
9,4466,AMERICAN VINTAGE BEVERAGE,2024-03-12,9033,2024-02-22,2024-04-16,147,1377.87,6.61,None


In [19]:
sales_4466 = pd.read_sql("select * from sales where VendorNo = 4466", conn)
sales_4466

,InventoryId,Store,Brand,Description,Size,SalesQuantity,SalesDollars,SalesPrice,SalesDate,Volume,Classification,ExciseTax,VendorNo,VendorName
0,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-09,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
1,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-12,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
2,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-15,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
3,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-21,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
4,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-23,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9448,9_BLACKPOOL_5215,9,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-12-21,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
9449,9_BLACKPOOL_5255,9,5255,TGI Fridays Ultimte Mudslide,1.75L,1,12.99,12.99,2024-12-02,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
9450,9_BLACKPOOL_5255,9,5255,TGI Fridays Ultimte Mudslide,1.75L,1,12.99,12.99,2024-12-09,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
9451,9_BLACKPOOL_5255,9,5255,TGI Fridays Ultimte Mudslide,1.75L,1,12.99,12.99,2024-12-23,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE


In [20]:
#Lets analyse the total sales by VendorNo = 4466 for each brand
sales_4466.groupby('Brand')[["SalesDollars","SalesQuantity","ExciseTax"]].sum()


,SalesDollars,SalesQuantity,ExciseTax
Brand,,,
3140,50531.10,3890,7149.25
5215,60416.49,4651,8548.96
5255,79187.04,6096,11204.28


In [21]:
#Let's also check sales for all vendors
Sales_by_venodors = pd.read_sql("SELECT VendorNo,VendorName, SUM(SalesDollars) as total_sales FROM sales GROUP BY VendorNo,VendorName order by total_sales desc ", conn)
Sales_by_venodors ['total_sale_millions'] = (Sales_by_venodors ['total_sales'] / 1_000_000).round(2)
Sales_by_venodors


,VendorNo,VendorName,total_sales,total_sale_millions
0,3960,DIAGEO NORTH AMERICA INC,6.874242e+07,68.74
1,4425,MARTIGNETTI COMPANIES,4.099240e+07,40.99
2,17035,PERNOD RICARD USA,3.228125e+07,32.28
3,12546,JIM BEAM BRANDS COMPANY,3.190632e+07,31.91
4,480,BACARDI USA INC,2.501456e+07,25.01
...,...,...,...,...
125,1439,CAPSTONE INTERNATIONAL,2.468700e+02,0.00
126,90034,EXCLUSIVE WINES & SPIRITS,5.598000e+01,0.00
127,9710,WHYTE & MACKAY,3.198000e+01,0.00
128,1002,BERNIKO LLC,1.699000e+01,0.00



**EDA Observations**

* **Purchases Table**: Detailed transactions with purchase_date, brand,
amount_paid (USD), and quantity. Suitable for time-series and vendor performance analysis.
* **Purchase Prices Table**: Product-level pricing for margin and cost optimization.
* **Vendor Invoice Table**: Aggregated purchases by vendor and PO; includes freight for cost tracking.
* **Sales Table**: Sales data with brand, quantity_sold, selling_price, and revenue. Supports profitability, demand forecasting, and pricing analysis.

* **begin_inventory** and **end_inventory** columns can be ignored as they are not required for our business problem.


---


Since the data required for analysis is spread across multiple tables, we will create a consolidated summary table that includes:

* Vendor purchase transactions
* Sales transaction details
* Freight costs per vendor
* Actual product prices from vendors

Before building the consolidated summary table, let’s review each summary table individually.



---




In [22]:
purchase_prices_4466.columns

Index(['Brand', 'Description', 'Price', 'Size', 'Volume', 'Classification',
       'PurchasePrice', 'VendorNumber', 'VendorName'],
      dtype='object')

In [23]:
purchases_4466.columns

Index(['InventoryId', 'Store', 'Brand', 'Description', 'Size', 'VendorNumber',
       'VendorName', 'PONumber', 'PODate', 'ReceivingDate', 'InvoiceDate',
       'PayDate', 'PurchasePrice', 'Quantity', 'Dollars', 'Classification'],
      dtype='object')

In [24]:
#freight_summary
freight_summary = pd.read_sql_query("""SELECT
    VendorNumber,
    SUM(Freight) as FreightCost
FROM vendor_invoice
GROUP BY VendorNumber""", conn)
freight_summary

,VendorNumber,FreightCost
0,2,27.08
1,54,0.48
2,60,367.52
3,105,62.39
4,200,6.19
...,...,...
121,98450,856.02
122,99166,130.09
123,172662,178.34
124,173357,202.50


In [25]:
#purchase_summary
purchase_summary = pd.read_sql_query("""select
              p.VendorNumber,
              p.VendorName,
              p.Brand,
              p.PurchasePrice,
              pp.Volume,
              pp.Price as actul_price,
              sum(p.Quantity) as total_purchase_quantity,
              sum(p.Dollars) as total_purchase_dollars
              from purchases p join purchase_prices pp
              on p.Brand = pp.Brand
              where p.PurchasePrice > 0
              group by p.VendorNumber, p.VendorName, p.Brand
              order by total_purchase_dollars""",conn)
purchase_summary

,VendorNumber,VendorName,Brand,PurchasePrice,Volume,actul_price,total_purchase_quantity,total_purchase_dollars
0,7245,PROXIMO SPIRITS INC.,3065,0.71,50,0.99,1,0.71
1,3960,DIAGEO NORTH AMERICA INC,6127,1.47,200,1.99,1,1.47
2,3924,HEAVEN HILL DISTILLERIES,9123,0.74,50,0.99,2,1.48
3,8004,SAZERAC CO INC,5683,0.39,50,0.49,6,2.34
4,9815,WINE GROUP INC,8527,1.32,750,4.99,2,2.64
...,...,...,...,...,...,...,...,...
10687,3960,DIAGEO NORTH AMERICA INC,3545,21.89,1750,29.99,138109,3023206.01
10688,3960,DIAGEO NORTH AMERICA INC,4261,16.17,1750,22.99,201682,3261197.94
10689,17035,PERNOD RICARD USA,8068,18.24,1750,24.99,187407,3418303.68
10690,4425,MARTIGNETTI COMPANIES,3405,23.19,1750,28.99,164038,3804041.22


In [27]:
#sales_summury
sales_summury = pd.read_sql("""SELECT
                    VendorNo,
                    VendorName,
                    Brand,
                    SUM(SalesDollars) as total_sales_dollars,
                    SUM(SalesPrice) as total_sales_price,
                    SUM(SalesQuantity) as total_sales_Quantity,
                    SUM(ExciseTax) as total_ExciseTax
                    FROM Sales
                    GROUP BY VendorNo,VendorName,Brand
                    order by total_sales_dollars desc """, conn)
sales_summury

,VendorNo,VendorName,Brand,total_sales_dollars,total_sales_price,total_sales_Quantity,total_ExciseTax
0,1128,BROWN-FORMAN CORP,1233,5.101920e+06,672819.31,142049,260999.20
1,4425,MARTIGNETTI COMPANIES,3405,4.819073e+06,561512.37,160247,294438.66
2,17035,PERNOD RICARD USA,8068,4.538121e+06,461140.15,187140,343854.07
3,3960,DIAGEO NORTH AMERICA INC,4261,4.475973e+06,420050.01,200412,368242.80
4,3960,DIAGEO NORTH AMERICA INC,3545,4.223108e+06,545778.28,135838,249587.83
...,...,...,...,...,...,...,...
11267,3252,E & J GALLO WINERY,3933,1.980000e+00,0.99,2,0.10
11268,3924,HEAVEN HILL DISTILLERIES,9123,1.980000e+00,0.99,2,0.10
11269,10050,Russian Standard Vodka,3623,1.980000e+00,1.98,2,0.10
11270,9206,PHILLIPS PRODUCTS CO.,2773,9.900000e-01,0.99,1,0.05


In [28]:
#Merge all summary tables into a single aggregated table for comprehensive analysis
vendor_sales_summary = pd.read_sql("""WITH
purchase_summary as (select
              p.VendorNumber,
              p.VendorName,
              p.Description,
              p.Brand,
              p.PurchasePrice,
              pp.Volume,
              pp.Price as actul_price,
              sum(p.Quantity) as total_purchase_quantity,
              sum(p.Dollars) as total_purchase_dollars
              from purchases p join purchase_prices pp
              on p.Brand = pp.Brand
              where p.PurchasePrice > 0
              group by p.VendorNumber, p.VendorName, p.Brand
              order by total_purchase_dollars),

  sales_summury as (SELECT
                    VendorNo,
                    VendorName,
                    Brand,
                    SUM(SalesDollars) as total_sales_dollars,
                    SUM(SalesPrice) as total_sales_price,
                    SUM(SalesQuantity) as total_sales_Quantity,
                    SUM(ExciseTax) as total_ExciseTax
                    FROM Sales
                    GROUP BY VendorNo,VendorName,Brand
                    order by total_sales_dollars),

freight_summary as (SELECT
        VendorNumber,
        SUM(Freight) as FreightCost
        FROM vendor_invoice
        GROUP BY VendorNumber)

select
        ps.VendorNumber,
        ps.VendorName,
        ps.Brand,
        ps.Description,
        ps.PurchasePrice,
        ps.Volume,
        ps.actul_price,
        ps.total_purchase_quantity,
        ps.total_purchase_dollars,
        ss.total_sales_dollars,
        ss.total_sales_price,
        ss.total_sales_Quantity,
        ss.total_ExciseTax,
        fs.FreightCost
        from purchase_summary ps
        left join sales_summury ss
            on ps.VendorNumber = ss.VendorNo and ps.Brand = ss.Brand
        left join freight_summary fs
            on ps.VendorNumber = fs.VendorNumber
        order by ps.total_purchase_dollars desc""",conn)





In [29]:
vendor_sales_summary

,VendorNumber,VendorName,Brand,Description,PurchasePrice,Volume,actul_price,total_purchase_quantity,total_purchase_dollars,total_sales_dollars,total_sales_price,total_sales_Quantity,total_ExciseTax,FreightCost
0,1128,BROWN-FORMAN CORP,1233,Jack Daniels No 7 Black,26.27,1750,36.99,145080,3811251.60,5.101920e+06,672819.31,142049.0,260999.20,68601.68
1,4425,MARTIGNETTI COMPANIES,3405,Tito's Handmade Vodka,23.19,1750,28.99,164038,3804041.22,4.819073e+06,561512.37,160247.0,294438.66,144929.24
2,17035,PERNOD RICARD USA,8068,Absolut 80 Proof,18.24,1750,24.99,187407,3418303.68,4.538121e+06,461140.15,187140.0,343854.07,123780.22
3,3960,DIAGEO NORTH AMERICA INC,4261,Capt Morgan Spiced Rum,16.17,1750,22.99,201682,3261197.94,4.475973e+06,420050.01,200412.0,368242.80,257032.07
4,3960,DIAGEO NORTH AMERICA INC,3545,Ketel One Vodka,21.89,1750,29.99,138109,3023206.01,4.223108e+06,545778.28,135838.0,249587.83,257032.07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10687,9815,WINE GROUP INC,8527,Concannon Glen Ellen Wh Zin,1.32,750,4.99,2,2.64,1.595000e+01,10.96,5.0,0.55,27100.41
10688,8004,SAZERAC CO INC,5683,Dr McGillicuddy's Apple Pie,0.39,50,0.49,6,2.34,6.566000e+01,1.47,134.0,7.04,50293.62
10689,3924,HEAVEN HILL DISTILLERIES,9123,Deep Eddy Vodka,0.74,50,0.99,2,1.48,1.980000e+00,0.99,2.0,0.10,14069.87
10690,3960,DIAGEO NORTH AMERICA INC,6127,The Club Strawbry Margarita,1.47,200,1.99,1,1.47,1.432800e+02,77.61,72.0,15.12,257032.07




---


This query generates a vendor-wise summary of sales and purchases, It is essential for the following reasons:


* Performance Optimization:

  * Involves heavy joins and aggregations on large datasets (sales and purchases).
  * Pre-aggregating results reduces repeated expensive computations.



* Business Insights:

  * Enables analysis of sales, purchases, and pricing across vendors and brands.



* Future Benefits:

  * Storing this data accelerates dashboarding and reporting.
  * Dashboards can fetch data directly from vendor_sales_summary instead of running complex queries each time.



Next Step:
Clean the data to resolve any inconsistencies.


---



In [30]:
#Data Types of columns
vendor_sales_summary.dtypes

,0
VendorNumber,int64
VendorName,object
Brand,int64
Description,object
PurchasePrice,float64
Volume,object
actul_price,float64
total_purchase_quantity,int64
total_purchase_dollars,float64
total_sales_dollars,float64


In [31]:
#check if there are any null values
vendor_sales_summary.isna().sum()

,0
VendorNumber,0
VendorName,0
Brand,0
Description,0
PurchasePrice,0
Volume,0
actul_price,0
total_purchase_quantity,0
total_purchase_dollars,0
total_sales_dollars,178


In [32]:
#Check for unnecessary "blank spaces"
vendor_sales_summary['VendorName'].unique()

array(['BROWN-FORMAN CORP          ', 'MARTIGNETTI COMPANIES',
       'PERNOD RICARD USA          ', 'DIAGEO NORTH AMERICA INC   ',
       'BACARDI USA INC            ', 'JIM BEAM BRANDS COMPANY    ',
       'MAJESTIC FINE WINES        ', 'ULTRA BEVERAGE COMPANY LLP ',
       'STOLI GROUP,(USA) LLC      ', 'PROXIMO SPIRITS INC.       ',
       'MOET HENNESSY USA INC      ', 'CAMPARI AMERICA            ',
       'SAZERAC CO INC             ', 'CONSTELLATION BRANDS INC   ',
       'M S WALKER INC             ', 'SAZERAC NORTH AMERICA INC. ',
       'PALM BAY INTERNATIONAL INC ', 'REMY COINTREAU USA INC     ',
       'SIDNEY FRANK IMPORTING CO  ', 'E & J GALLO WINERY         ',
       'WILLIAM GRANT & SONS INC   ', 'HEAVEN HILL DISTILLERIES   ',
       'DISARONNO INTERNATIONAL LLC', 'EDRINGTON AMERICAS         ',
       'CASTLE BRANDS CORP.        ', 'SOUTHERN WINE & SPIRITS NE ',
       'STE MICHELLE WINE ESTATES  ', 'TRINCHERO FAMILY ESTATES   ',
       'MHW LTD                    ', 'W

In [33]:
vendor_sales_summary['Volume'].unique()

array(['1750', '750', '1000', '1500', '375', '50', '3000', '5000', '100',
       '200', '4000', '187', '500', '600', '300', '1100', '250', '400',
       '18000', '150', '720', '330', '162.5', '180', '19500', '6000',
       '560', '20000', '9000', '3750', '650'], dtype=object)

**Observations**

1. The Volume column is numeric but stored as an object type.
2. Some products have missing values because they were not sold.
3. Categorical columns contain unwanted white spaces.

In [34]:
# changing datatype to float
vendor_sales_summary['Volume'] = vendor_sales_summary['Volume'].astype('float')

# filling missing value with 0
vendor_sales_summary.fillna(0,inplace = True)

# removing spaces from categorical columns
vendor_sales_summary['VendorName'] = vendor_sales_summary['VendorName'].str.strip()
vendor_sales_summary['Description'] = vendor_sales_summary['Description'].str.strip()

In [35]:
vendor_sales_summary.columns

Index(['VendorNumber', 'VendorName', 'Brand', 'Description', 'PurchasePrice',
       'Volume', 'actul_price', 'total_purchase_quantity',
       'total_purchase_dollars', 'total_sales_dollars', 'total_sales_price',
       'total_sales_Quantity', 'total_ExciseTax', 'FreightCost'],
      dtype='object')

In [36]:
df = vendor_sales_summary

# 1. Profit-related Features
df['total_profit'] = df['total_sales_dollars'] - df['total_purchase_dollars']

# 2. Ratios & Percentages
df['sales_purchase_ratio'] = df['total_sales_dollars'] / df['total_purchase_dollars']

# 3. Cost Analysis
df['excise_tax_per_unit'] = df['total_ExciseTax'] / df['total_sales_Quantity']
df['freight_per_unit'] = df['FreightCost'] / df['total_purchase_quantity']
df['total_cost_per_unit'] = df['PurchasePrice'] + df['excise_tax_per_unit'] + df['freight_per_unit']

# 4. Revenue Metrics
df['avg_selling_price'] = df['total_sales_dollars'] / df['total_sales_Quantity']
df['avg_purchase_price'] = df['total_purchase_dollars'] / df['total_purchase_quantity']

vendor_sales_summary = df


In [37]:
vendor_sales_summary

,VendorNumber,VendorName,Brand,Description,PurchasePrice,Volume,actul_price,total_purchase_quantity,total_purchase_dollars,total_sales_dollars,...,total_sales_Quantity,total_ExciseTax,FreightCost,total_profit,sales_purchase_ratio,excise_tax_per_unit,freight_per_unit,total_cost_per_unit,avg_selling_price,avg_purchase_price
0,1128,BROWN-FORMAN CORP,1233,Jack Daniels No 7 Black,26.27,1750.0,36.99,145080,3811251.60,5.101920e+06,...,142049.0,260999.20,68601.68,1290667.91,1.338647,1.837389,0.472854,28.580243,35.916617,26.27
1,4425,MARTIGNETTI COMPANIES,3405,Tito's Handmade Vodka,23.19,1750.0,28.99,164038,3804041.22,4.819073e+06,...,160247.0,294438.66,144929.24,1015032.27,1.266830,1.837405,0.883510,25.910915,30.072784,23.19
2,17035,PERNOD RICARD USA,8068,Absolut 80 Proof,18.24,1750.0,24.99,187407,3418303.68,4.538121e+06,...,187140.0,343854.07,123780.22,1119816.92,1.327594,1.837416,0.660489,20.737905,24.249870,18.24
3,3960,DIAGEO NORTH AMERICA INC,4261,Capt Morgan Spiced Rum,16.17,1750.0,22.99,201682,3261197.94,4.475973e+06,...,200412.0,368242.80,257032.07,1214774.94,1.372493,1.837429,1.274442,19.281871,22.333857,16.17
4,3960,DIAGEO NORTH AMERICA INC,3545,Ketel One Vodka,21.89,1750.0,29.99,138109,3023206.01,4.223108e+06,...,135838.0,249587.83,257032.07,1199901.61,1.396897,1.837393,1.861081,25.588475,31.089295,21.89
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10687,9815,WINE GROUP INC,8527,Concannon Glen Ellen Wh Zin,1.32,750.0,4.99,2,2.64,1.595000e+01,...,5.0,0.55,27100.41,13.31,6.041667,0.110000,13550.205000,13551.635000,3.190000,1.32
10688,8004,SAZERAC CO INC,5683,Dr McGillicuddy's Apple Pie,0.39,50.0,0.49,6,2.34,6.566000e+01,...,134.0,7.04,50293.62,63.32,28.059829,0.052537,8382.270000,8382.712537,0.490000,0.39
10689,3924,HEAVEN HILL DISTILLERIES,9123,Deep Eddy Vodka,0.74,50.0,0.99,2,1.48,1.980000e+00,...,2.0,0.10,14069.87,0.50,1.337838,0.050000,7034.935000,7035.725000,0.990000,0.74
10690,3960,DIAGEO NORTH AMERICA INC,6127,The Club Strawbry Margarita,1.47,200.0,1.99,1,1.47,1.432800e+02,...,72.0,15.12,257032.07,141.81,97.469388,0.210000,257032.070000,257033.750000,1.990000,1.47


# **Saving Cleaned and consolidated summary table into database**

In [38]:
cursor = conn.cursor()

In [39]:
# This query will only runs once
cursor.execute("""CREATE TABLE vendor_sales_summary (
    VendorNumber INT,
    VendorName VARCHAR(100),
    Brand INT,
    Description VARCHAR(100),
    PurchasePrice DECIMAL(10,2),
    ActualPrice DECIMAL(10,2),
    Volume INT,
    TotalPurchaseQuantity INT,
    TotalPurchaseDollars DECIMAL(15,2),
    TotalSalesQuantity INT,
    TotalSalesDollars DECIMAL(15,2),
    TotalSalesPrice DECIMAL(15,2),
    TotalExciseTax DECIMAL(15,2),
    FreightCost DECIMAL(15,2),
    GrossProfit DECIMAL(15,2),
    ProfitMargin DECIMAL(15,2),
    StockTurnover DECIMAL(15,2),
    SalesToPurchaseRatio DECIMAL(15,2),
    PRIMARY KEY (VendorNumber, Brand)
);
""")

In [40]:
vendor_sales_summary.to_sql('vendor_sales_summary', conn, if_exists = 'replace', index = False )

10692

In [41]:
pd.read_sql_query("select * from vendor_sales_summary",conn)

,VendorNumber,VendorName,Brand,Description,PurchasePrice,Volume,actul_price,total_purchase_quantity,total_purchase_dollars,total_sales_dollars,...,total_sales_Quantity,total_ExciseTax,FreightCost,total_profit,sales_purchase_ratio,excise_tax_per_unit,freight_per_unit,total_cost_per_unit,avg_selling_price,avg_purchase_price
0,1128,BROWN-FORMAN CORP,1233,Jack Daniels No 7 Black,26.27,1750.0,36.99,145080,3811251.60,5.101920e+06,...,142049.0,260999.20,68601.68,1290667.91,1.338647,1.837389,0.472854,28.580243,35.916617,26.27
1,4425,MARTIGNETTI COMPANIES,3405,Tito's Handmade Vodka,23.19,1750.0,28.99,164038,3804041.22,4.819073e+06,...,160247.0,294438.66,144929.24,1015032.27,1.266830,1.837405,0.883510,25.910915,30.072784,23.19
2,17035,PERNOD RICARD USA,8068,Absolut 80 Proof,18.24,1750.0,24.99,187407,3418303.68,4.538121e+06,...,187140.0,343854.07,123780.22,1119816.92,1.327594,1.837416,0.660489,20.737905,24.249870,18.24
3,3960,DIAGEO NORTH AMERICA INC,4261,Capt Morgan Spiced Rum,16.17,1750.0,22.99,201682,3261197.94,4.475973e+06,...,200412.0,368242.80,257032.07,1214774.94,1.372493,1.837429,1.274442,19.281871,22.333857,16.17
4,3960,DIAGEO NORTH AMERICA INC,3545,Ketel One Vodka,21.89,1750.0,29.99,138109,3023206.01,4.223108e+06,...,135838.0,249587.83,257032.07,1199901.61,1.396897,1.837393,1.861081,25.588475,31.089295,21.89
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10687,9815,WINE GROUP INC,8527,Concannon Glen Ellen Wh Zin,1.32,750.0,4.99,2,2.64,1.595000e+01,...,5.0,0.55,27100.41,13.31,6.041667,0.110000,13550.205000,13551.635000,3.190000,1.32
10688,8004,SAZERAC CO INC,5683,Dr McGillicuddy's Apple Pie,0.39,50.0,0.49,6,2.34,6.566000e+01,...,134.0,7.04,50293.62,63.32,28.059829,0.052537,8382.270000,8382.712537,0.490000,0.39
10689,3924,HEAVEN HILL DISTILLERIES,9123,Deep Eddy Vodka,0.74,50.0,0.99,2,1.48,1.980000e+00,...,2.0,0.10,14069.87,0.50,1.337838,0.050000,7034.935000,7035.725000,0.990000,0.74
10690,3960,DIAGEO NORTH AMERICA INC,6127,The Club Strawbry Margarita,1.47,200.0,1.99,1,1.47,1.432800e+02,...,72.0,15.12,257032.07,141.81,97.469388,0.210000,257032.070000,257033.750000,1.990000,1.47


# **Conclusion of EDA**

* We have created a consolidated summary table that combines purchase and sales data for all vendors.
* This summary table has been inserted into the database, making it readily available for downstream analysis and reporting.
* The table serves as a single source of truth, reducing the need for repeated complex queries and improving performance.
* It enables faster dashboarding, business insights, and supports future analytics use cases such as trend analysis, vendor performance evaluation, and pricing strategies.
* Data quality checks have been performed, and any inconsistencies have been addressed to ensure accuracy and reliability.

